# Phase 5 — Interactive Gradio Demo + Pre-Sigmoid Logit Grad-CAM

Notebook peluncur ringan (*launcher*) di Google Colab untuk menjalankan **demo interaktif** klasifikasi **Real Art vs AI-Generated Art** (`app.py`) menggunakan dua model produksi hasil ekspor Phase 5:
- **Primary Champion (`Exp-3B`):** `art_classifier_resnet50_ft.keras` (`91.61 MB`, Test Acc `98.58%`, OOD Acc `90.00%`)
- **Pareto Lightweight (`Exp-2B`):** `art_classifier_mobilenetv2_ft.keras` (`9.80 MB`, Test Acc `97.87%`, `9.3×` lebih kecil)

In [ ]:
# Cell 1 — Install Gradio pada runtime Colab
!pip install -q gradio

In [ ]:
# Cell 2 — Mount Google Drive & Sinkronisasi Repo GitHub
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Indraku19/real-art-vs-ai-generated-art.git'
COLAB_DIR = Path('/content/real-art-vs-ai-generated-art')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not COLAB_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(COLAB_DIR), 'pull', '--ff-only'])
    os.chdir(COLAB_DIR)
print('Working dir:', Path.cwd())

In [ ]:
# Cell 3 — Hubungkan models/export/ ke Google Drive 04_Models/
import shutil
from pathlib import Path

EXPORT_MODELS = [
    'art_classifier_resnet50_ft.keras',
    'art_classifier_mobilenetv2_ft.keras',
]
drive_models_dir = Path('/content/drive/MyDrive/Real-vs-AI-Art-Classification/04_Models')
local_export_dir = Path('models/export')
local_export_dir.mkdir(parents=True, exist_ok=True)

for fname in EXPORT_MODELS:
    dst = local_export_dir / fname
    src = drive_models_dir / fname
    if not dst.exists():
        if not src.exists():
            raise FileNotFoundError(f'Exported model not found in Drive: {src}')
        print(f'Copying {fname} from Drive...')
        shutil.copy2(src, dst)
    print(f'[OK] {dst} ({dst.stat().st_size / (1024**2):.2f} MB)')

In [ ]:
# Cell 4 — Verifikasi Muat Model + Smoke Test Grad-CAM Lintas-Versi Keras
from PIL import Image
import numpy as np
from src.inference import ArtClassifierEngine

engine = ArtClassifierEngine()
dummy_img = Image.fromarray(np.full((256, 256, 3), 128, dtype=np.uint8))
for key in ('resnet50_ft', 'mobilenetv2_ft'):
    out = engine.predict_with_gradcam(dummy_img, model_key=key)
    print(f'[VERIFIED] {key} -> pred={out["predicted_label"]} P(AI)={out["prob_ai"]:.4f} ({out["latency_ms"]} ms)')

In [ ]:
# Cell 5 — Luncurkan Demo Interaktif Gradio (--share untuk link publik Colab)
!python app.py --share --port 7860